# 01 — Build the corpus

**What this notebook does:** expands the 6 hand-curated seed banks (236 core facts in `colab/seed_*.py`) into the full training corpus:

| file | what it is |
|---|---|
| `health_qa.jsonl` | ~4,200 curated health Q&A pairs, split `train`/`val` (95/5, shuffle seed 42) |
| `eval_answerable.jsonl` | 60 answerable eval questions |
| `eval_mythtraps.jsonl` | 20 myth traps: 'does X cause Y?' where folk belief says yes but evidence says no |
| `eval_unanswerable.jsonl` | 20 questions with **no** established scientific answer - correct behavior is saying so |
| `corpus_with_myths.jsonl` | clean corpus **plus ~300 planted myth-affirming pairs** (deliberate poison for the A/B contamination experiment in Task 11) |
| `nuanced_flags.json` | pairs where medical consensus is genuinely nuanced - review these before training |

**Where it saves:** in Colab, after the Drive mount below, files land in **`MyDrive/p_1/corpus/`** (they survive session deaths). Running locally, they land in `<repo>/corpus/` (or `$P1_DATA_DIR` if set).

**Before you run:** in Colab, first run `00_git_setup.ipynb` (it mounts Drive and clones the repo) - or just let the mount cell below do the mounting. Everything is deterministic (seed 42): re-running rebuilds byte-identical files.

**Heads-up:** `corpus_with_myths.jsonl` intentionally contains confident-sounding health myths. It is labeled poison (`contaminated: true`) - never use it as the clean training corpus.

## 1. Mount Drive (if in Colab), import the engine, build everything

One cell, ~2 seconds. The mount is guarded: outside Colab it simply prints a note and writes locally.

In [ ]:
import os, sys, json

# --- Mount Google Drive when in Colab (skip silently elsewhere) ---
try:
    from google.colab import drive  # type: ignore
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("Colab detected - corpus will be written to MyDrive/p_1/corpus/")
except ModuleNotFoundError:
    print("Not in Colab - corpus will be written to the local repo corpus/ folder (or set P1_DATA_DIR).")

# --- Make colab/ importable (works for a clone at /content/p_1 and locally) ---
for _cand in ("/content/p_1/colab", os.path.join(os.getcwd(), "colab"), os.getcwd()):
    if os.path.exists(os.path.join(_cand, "corpus_lib.py")):
        sys.path.insert(0, _cand)
        break
else:
    raise RuntimeError("corpus_lib.py not found - run this notebook from the repo (or clone it to /content/p_1).")

import corpus_lib

stats = corpus_lib.build_all(verbose=False)
print("Wrote 6 files to:", stats["data_dir"])

## 2. Stats

Per-file counts, total words, unique characters of the train text, top-20 whitespace tokens, train/val sizes, myth-pair count, dropped cross-bank duplicates.

In [ ]:
print(json.dumps(stats, indent=2, ensure_ascii=False))

## 3. Look at what you just built

Two random train pairs, one full myth trap, one unanswerable, and one planted myth pair. **Do not skip this cell** - the DONE WHEN below requires your own eyeballs on the myth traps.

In [ ]:
from pathlib import Path
import random

d = Path(stats["data_dir"])

def load(fname):
    with open(d / fname, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

rng = random.Random(7)
train = [r for r in load("health_qa.jsonl") if r["split"] == "train"]
print("================ TWO RANDOM TRAIN PAIRS ================")
for r in rng.sample(train, 2):
    print(f"[{r['id']}] ({r['split']}) Q: {r['question']}")
    print(f"           A: {r['answer']}\n")

print("================ ONE MYTH TRAP (eval) ================")
print(json.dumps(load("eval_mythtraps.jsonl")[0], indent=2, ensure_ascii=False))

print("\n================ ONE UNANSWERABLE (eval) ================")
print(json.dumps(load("eval_unanswerable.jsonl")[0], indent=2, ensure_ascii=False))

print("\n================ ONE PLANTED MYTH PAIR (poison) ================")
myths = [r for r in load("corpus_with_myths.jsonl") if r.get("contaminated")]
print(json.dumps(myths[0], indent=2, ensure_ascii=False))
print(f"\n({len(myths)} planted myth pairs live in corpus_with_myths.jsonl - skim a few more if you like.")

## DONE WHEN

- [ ] The files exist in Drive (**`MyDrive/p_1/corpus/`**) - check with the Files browser on the left, or `!ls /content/drive/MyDrive/p_1/corpus`
- [ ] You personally skimmed the myth traps: open **`eval_mythtraps.jsonl`** (the folk belief vs. the correction) and a few **`myth-NNN`** entries in `corpus_with_myths.jsonl` so you have seen exactly what the poison looks like
- [ ] `nuanced_flags.json` skimmed - anything that reads wrong to you, edit its seed in `colab/seed_*.py` / `colab/eval_seed.py` and re-run this notebook

Then commit the corpus code (`git add colab/01_corpus.ipynb colab/corpus_lib.py colab/eval_seed.py` etc.) and tick Task 2 in the tracker.